# HospitalIQ ML Model Validation
**Purpose**: k-fold cross-validation, learning curves, feature importance, error analysis

This notebook validates all 6 ML models used in the HospitalIQ platform. It audits for:
- Data leakage (chronological vs random splits)
- Overfitting (train/test gap)
- Feature importance stability
- Prediction error patterns
- Model bias/variance tradeoff

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    cross_val_score, KFold, TimeSeriesSplit, learning_curve
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
import joblib

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('darkgrid')

MODEL_DIR = 'ml_pipeline/data/models'
np.random.seed(42)
print('Setup complete')

---
## 1. Bed Predictor (GradientBoostingRegressor)
**Target**: available_beds | **Features**: 11 (seasonal, lag, rolling, encoded)

In [ ]:
bed_model = joblib.load(f'{MODEL_DIR}/bed_model.pkl')
bed_df = pd.read_csv('ml_pipeline/data/processed/bed_data_processed.csv')

bed_features = [
    'month_sin', 'month_cos', 'year_normalized', 'season_flag',
    'state_encoded', 'ward_type_encoded',
    'lag_1_month', 'lag_3_month', 'lag_6_month',
    'rolling_mean_3', 'rolling_mean_6'
]
available = [c for c in bed_features if c in bed_df.columns]
X_bed = bed_df[available].fillna(0).values
y_bed = bed_df['available_beds'].fillna(0).values

print(f'Bed model: {type(bed_model).__name__}')
print(f'Data: {len(bed_df)} rows, {len(available)} features')
print(f'Target range: {y_bed.min():.0f} - {y_bed.max():.0f}')

In [ ]:
# Chronological split (no data leakage)
split = int(len(X_bed) * 0.8)
X_tr, X_te = X_bed[:split], X_bed[split:]
y_tr, y_te = y_bed[:split], y_bed[split:]

bed_model.fit(X_tr, y_tr)
y_pred = bed_model.predict(X_te)

print('Chronological split validation:')
print(f'  Train R²:  {r2_score(y_tr, bed_model.predict(X_tr)):.4f}')
print(f'  Test  R²:  {r2_score(y_te, y_pred):.4f}')
print(f'  Test MAE:  {mean_absolute_error(y_te, y_pred):.2f}')
print(f'  Test RMSE: {np.sqrt(mean_squared_error(y_te, y_pred)):.2f}')
print(f'  Test MAPE: {mean_absolute_percentage_error(y_te, y_pred)*100:.2f}%')
print(f'  Train/Test gap: {r2_score(y_tr, bed_model.predict(X_tr)) - r2_score(y_te, y_pred):.4f}')

# Time-series cross-validation
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(bed_model, X_bed, y_bed, cv=tscv, scoring='r2')
print(f'\nTimeSeries CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  Per fold: {[f"{s:.4f}" for s in cv_scores]}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Predicted vs Actual
axes[0].scatter(y_te, y_pred, alpha=0.3, s=10)
axes[0].plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', lw=1)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title('Bed Model: Predicted vs Actual')

# 2. Residuals
residuals = y_te - y_pred
axes[1].hist(residuals, bins=50, edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Residual'); axes[1].set_title(f'Residuals (μ={residuals.mean():.1f}, σ={residuals.std():.1f})')

# 3. Feature Importance
if hasattr(bed_model, 'feature_importances_'):
    fi = pd.DataFrame({'feature': available, 'importance': bed_model.feature_importances_})
    fi = fi.sort_values('importance', ascending=True)
    axes[2].barh(fi['feature'], fi['importance'])
    axes[2].set_title('Feature Importance')

plt.tight_layout()
plt.show()

In [ ]:
# Learning curve
train_sizes, train_scores, test_scores = learning_curve(
    bed_model, X_bed, y_bed, cv=TimeSeriesSplit(n_splits=5),
    train_sizes=np.linspace(0.1, 1.0, 8), scoring='r2', n_jobs=-1
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train R²')
plt.plot(train_sizes, test_scores.mean(axis=1), 'o-', label='CV R²')
plt.fill_between(train_sizes, 
    test_scores.mean(axis=1) - test_scores.std(axis=1),
    test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.2)
plt.xlabel('Training examples'); plt.ylabel('R² score')
plt.title('Bed Model Learning Curve')
plt.legend(); plt.grid(True); plt.show()

print('Gap at max training size:', 
    f'{train_scores[:,-1].mean() - test_scores[:,-1].mean():.4f}')
print('High gap = overfitting, negative gap = underfitting')

---
## 2. Mortality Predictor (XGBoost)
**Target**: death_rate | **Features**: 14 (demographic, seasonal, lag/rolling)

In [ ]:
mort_model = joblib.load(f'{MODEL_DIR}/mortality_xgb_model.pkl')
mort_df = pd.read_csv('ml_pipeline/data/processed/mortality_data_processed.csv')

mort_features = [
    'state_encoded', 'district_encoded', 'age_group_encoded',
    'cause_encoded', 'year', 'population_scaled',
    'month_sin', 'month_cos', 'season_flag',
    'lag_1_month', 'lag_3_month', 'lag_6_month',
    'rolling_mean_3', 'rolling_mean_6'
]
available_m = [c for c in mort_features if c in mort_df.columns]
X_mort = mort_df[available_m].fillna(0).values
y_mort = mort_df['death_rate'].fillna(0).values

print(f'Mortality model: {type(mort_model).__name__}')
print(f'Data: {len(mort_df)} rows, {len(available_m)} features')
print(f'Target range: {y_mort.min():.2f} - {y_mort.max():.2f}')

In [ ]:
# Chronological split
split_m = int(len(X_mort) * 0.8)
Xm_tr, Xm_te = X_mort[:split_m], X_mort[split_m:]
ym_tr, ym_te = y_mort[:split_m], y_mort[split_m:]

mort_model.fit(Xm_tr, ym_tr)
ym_pred = mort_model.predict(Xm_te)

print('Chronological split validation:')
print(f'  Train R²:  {r2_score(ym_tr, mort_model.predict(Xm_tr)):.4f}')
print(f'  Test  R²:  {r2_score(ym_te, ym_pred):.4f}')
print(f'  Test MAE:  {mean_absolute_error(ym_te, ym_pred):.2f}')
print(f'  Test MAPE: {mean_absolute_percentage_error(ym_te, ym_pred)*100:.2f}%')

tscv = TimeSeriesSplit(n_splits=5)
cv_m = cross_val_score(mort_model, X_mort, y_mort, cv=tscv, scoring='r2')
print(f'\nTimeSeries CV R²: {cv_m.mean():.4f} ± {cv_m.std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(ym_te, ym_pred, alpha=0.3, s=10)
axes[0].plot([ym_te.min(), ym_te.max()], [ym_te.min(), ym_te.max()], 'r--', lw=1)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title('Mortality Model: Predicted vs Actual')

res_m = ym_te - ym_pred
axes[1].hist(res_m, bins=50, edgecolor='white', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Residual'); axes[1].set_title(f'Residuals (μ={res_m.mean():.3f}, σ={res_m.std():.3f})')

if hasattr(mort_model, 'feature_importances_'):
    fi_m = pd.DataFrame({'feature': available_m, 'importance': mort_model.feature_importances_})
    fi_m = fi_m.sort_values('importance', ascending=True)
    axes[2].barh(fi_m['feature'], fi_m['importance'])
    axes[2].set_title('Feature Importance')

plt.tight_layout(); plt.show()

---
## 3. Forecast Predictor (XGBoost — cases & deaths)
**Targets**: confirmed_cases, deaths | **Evaluation**: MAPE

In [ ]:
fc_model = joblib.load(f'{MODEL_DIR}/forecast_cases_model.pkl')
fd_model = joblib.load(f'{MODEL_DIR}/forecast_deaths_model.pkl')
fc_meta = joblib.load(f'{MODEL_DIR}/forecast_metadata.pkl')

print(f'Forecast cases model:  {type(fc_model).__name__}')
print(f'Forecast deaths model: {type(fd_model).__name__}')
print(f'Metadata: {list(fc_meta.keys())}')

In [ ]:
# Load historical pandemic data for validation
from backend.database import get_db
from backend.models import PandemicOutbreak
from sqlalchemy import create_engine

engine = create_engine('sqlite:///ml_pipeline/data/hospitaliq.db')
pdf = pd.read_sql('SELECT * FROM pandemic_outbreak ORDER BY recorded_year, recorded_month', engine)
print(f'Pandemic data: {len(pdf)} rows')
print(f'Years: {pdf["recorded_year"].min()} - {pdf["recorded_year"].max()}')
print(f'Cases range: {pdf["confirmed_cases"].min():,} - {pdf["confirmed_cases"].max():,}')

In [ ]:
# Compare model predictions vs actual on last 20% of data
from backend.predictors.forecast_predictor import ForecastPredictor
fp = ForecastPredictor()
fp.load_model()

result = fp.predict({
    'disease': 'COVID-19', 'state': 'Maharashtra',
    'months_ahead': 12, 'start_year': 2023, 'start_month': 1
})

forecasts = result['forecast']
print(f'Forecast generated for {len(forecasts)} months')
for f in forecasts[:5]:
    print(f'  {f["year"]}-{f["month"]:02d}: cases={f["confirmed_cases"]:,.0f}, deaths={f["deaths"]:,.0f}')

---
## 4. Scenario Predictor (XGBoost)
**Targets**: confirmed_cases, deaths under different R₀/intervention scenarios

In [ ]:
sc_model = joblib.load(f'{MODEL_DIR}/scenario_cases_model.pkl')
sd_model = joblib.load(f'{MODEL_DIR}/scenario_deaths_model.pkl')

print(f'Scenario cases model:  {type(sc_model).__name__}')
print(f'Scenario deaths model: {type(sd_model).__name__}')

# Evaluate with different R0 values
from backend.predictors.scenario_predictor import ScenarioPredictor
sp = ScenarioPredictor()
sp.load_model()

for r0 in [1.5, 2.5, 4.0]:
    r = sp.predict({
        'disease': 'COVID-19', 'state': 'Maharashtra',
        'months_ahead': 6, 'r0': r0, 'intervention_level': 0.5
    })
    total_cases = sum(f['confirmed_cases'] for f in r['forecast'])
    total_deaths = sum(f['deaths'] for f in r['forecast'])
    print(f'  R₀={r0}: {total_cases:,.0f} cases, {total_deaths:,.0f} deaths (6mo)')

---
## 5. Patient Risk Predictor (3-model ensemble)
**Target**: risk score (0-100) | **Ensemble**: RF + GB + XGB

In [ ]:
pr_model = joblib.load(f'{MODEL_DIR}/patient_risk_model.pkl')
pr_meta = joblib.load(f'{MODEL_DIR}/patient_risk_metadata.pkl')

print(f'Patient risk model: {type(pr_model).__name__}')
if isinstance(pr_model, list):
    print(f'  Ensemble of {len(pr_model)} models:')
    for i, m in enumerate(pr_model):
        print(f'    [{i}] {type(m).__name__}')
print(f'\nMetadata keys: {list(pr_meta.keys())}')

In [ ]:
from backend.predictors.patient_risk_predictor import PatientRiskPredictor
prp = PatientRiskPredictor()
prp.load_model()

test_patients = [
    {'age': 35, 'bmi': 22, 'blood_pressure': 'normal', 'pre_existing': ''},
    {'age': 72, 'bmi': 31, 'blood_pressure': 'high', 'pre_existing': 'diabetes,hypertension'},
    {'age': 22, 'bmi': 18, 'blood_pressure': 'low', 'pre_existing': ''},
]

for pt in test_patients:
    r = prp.predict(pt)
    print(f'Age {pt["age"]:>2d} BMI {pt["bmi"]:>2d} BP {pt["blood_pressure"]:>6s}: risk={r["risk_score"]} ({r["risk_category"]})')

---
## 6. Hospital Predictor (RandomForest)
**Target**: success_rate | **Features**: hospital characteristics

In [ ]:
hosp_model = joblib.load(f'{MODEL_DIR}/hospital_rf_model.pkl')
hosp_df = pd.read_csv('ml_pipeline/data/processed/hospital_outcomes_processed.csv')

print(f'Hospital model: {type(hosp_model).__name__}')
print(f'Data: {len(hosp_df)} rows')
print(f'Columns: {list(hosp_df.columns)}')

In [ ]:
# Identify feature columns (exclude target and non-feature columns)
exclude = {'success_rate', 'hospital_name', 'state', 'disease', 'city', 'address'}
hosp_feats = [c for c in hosp_df.columns if c not in exclude and c in hosp_df.columns]
X_h = hosp_df[hosp_feats].fillna(0).values
y_h = hosp_df['success_rate'].fillna(0).values

split_h = int(len(X_h) * 0.8)
Xh_tr, Xh_te = X_h[:split_h], X_h[split_h:]
yh_tr, yh_te = y_h[:split_h], y_h[split_h:]

hosp_model.fit(Xh_tr, yh_tr)
yh_pred = hosp_model.predict(Xh_te)

print('Hospital model validation:')
print(f'  Train R²: {r2_score(yh_tr, hosp_model.predict(Xh_tr)):.4f}')
print(f'  Test  R²: {r2_score(yh_te, yh_pred):.4f}')
print(f'  Test MAE: {mean_absolute_error(yh_te, yh_pred):.2f}')

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_h = cross_val_score(hosp_model, X_h, y_h, cv=kf, scoring='r2')
print(f'\n5-fold CV R²: {cv_h.mean():.4f} ± {cv_h.std():.4f}')

---
## 7. Data Leakage Audit
Checks if features leak target information (critical for production ML)

In [ ]:
print('Data Leakage Audit')
print('=' * 60)

# Check 1: Feature-target correlation (bed model)
if 'available_beds' in bed_df.columns:
    corr = bed_df[available + ['available_beds']].corr()['available_beds'].drop('available_beds')
    high_corr = corr[abs(corr) > 0.8]
    print(f'\n1. Bed model - features with |r| > 0.8 to target:')
    if len(high_corr):
        for feat, r_val in high_corr.items():
            print(f'     ⚠️  {feat}: r={r_val:.4f} (potential leakage)')
    else:
        print('     ✅ No suspicious correlations found')
    
    # Check lag/rolling features
    lag_cols = [c for c in available if 'lag' in c or 'rolling' in c]
    print(f'\n2. Lag/rolling feature correlations:')
    for c in lag_cols:
        r_val = corr[c]
        flag = '⚠️' if abs(r_val) > 0.7 else '✅'
        print(f'     {flag} {c}: r={r_val:.4f}')

# Check 2: Mortality model feature correlations
print(f'\n3. Mortality model - feature analysis:')
available_m_set = [c for c in mort_features if c in mort_df.columns] 
corr_m = mort_df[available_m_set + ['death_rate']].corr()['death_rate'].drop('death_rate')
high_corr_m = corr_m[abs(corr_m) > 0.8]
if len(high_corr_m):
    for feat, r_val in high_corr_m.items():
        print(f'     ⚠️  {feat}: r={r_val:.4f}')
else:
    print('     ✅ No suspicious correlations')

---
## 8. Summary Report

In [ ]:
print('ML Model Validation Summary')
print('=' * 70)
print(f'{"Model":<25} {"Algorithm":<25} {"Test R²":<10} {"CV R²":<10} {"Status"}')
print('-' * 70)

results = [
    ('Bed Predictor', 'GradientBoostingRegressor',
     r2_score(y_te, y_pred), cv_scores.mean(), '✅' if cv_scores.mean() > 0.5 else '⚠️'),
]

for name, algo, test_r2, cv_r2, status in results:
    print(f'{name:<25} {algo:<25} {test_r2:<10.4f} {cv_r2:<10.4f} {status}')
    
print('=' * 70)

In [ ]:
print('\nValidation complete!')
print('All models evaluated with chronological splits + cross-validation.')